In [35]:
import pandas as pd 
import numpy as np 
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import random
from xgboost import XGBRegressor

In [36]:
simulation_data = pd.read_csv('../CafeteriaSimulation/cafeteriasimulationdata.csv')
simulation_data.head(5)

,minute,rate,arrivals,last_5_min,last_10_min,type,next_10_min,lunch_id
0,1,3,4,4.0,4.0,slow,35.0,slow1
1,2,3,4,8.0,8.0,slow,32.0,slow1
2,3,3,5,13.0,13.0,slow,33.0,slow1
3,4,3,1,14.0,14.0,slow,34.0,slow1
4,5,3,7,21.0,21.0,slow,32.0,slow1


### Setting up the simulating variables 

In [37]:
lunch_duration = 120 ## The lunch will run for 120 minutes 
cooking_time = 10 ## Total time required to cook the food 
batch_size = 20 ## Portions produced per batch 
available_portions = 20 ## Food currently ready to serve 
queue = 0 ## Nobody is waiting for the food for now. This is used to keep track of how many people are waiting 
cooking_minutes_remaining = 0 ## No batch is getting cooked currently 

### Main Goal of the Simulation: Finding out which category reduces students' waiting and food shortages while minimizing the leftover food. 

In [38]:
### Generating random students 
total_minutes = [] ## total minutes the lunch period runs
total_students_per_min = [] ## total students that came in the session
for i in range(lunch_duration): 
    total_minutes.append(i+1)
    total_students = random.randint(1,7)
    total_students_per_min.append(total_students)


In [39]:
### Input dataframe 
data = pd.DataFrame({
    "minutes": total_minutes,
    "arriving_students": total_students_per_min
})
data["last_5_min"] = (data["arriving_students"].rolling(5, min_periods=1).sum()) ## Cumulative sum for first 5
data["last_10_min"] = (data["arriving_students"].rolling(10, min_periods=1).sum()) ## Cumulative sum for first 10 

### Training a model 

In [40]:
model = XGBRegressor(objective = 'reg:squarederror', n_estimators = 200, max_depths = 3, learning_rate = 0.05, subsample= 0.8, colsample_bytree = 1.0, random_state =42)

### Splitting the data into train test split in order to train and test the data 

In [42]:
simulation_data_c = simulation_data.copy()
## Since there are 100 samples of each lunch_id we will grab only one of each 
lunch_info = simulation_data_c[['lunch_id', 'type']].drop_duplicates()
train_ids, test_ids = train_test_split(lunch_info['lunch_id'], test_size =0.2, stratify=lunch_info["type"], random_state=42)
train = simulation_data_c[simulation_data_c['lunch_id'].isin(train_ids)]
test = simulation_data_c[simulation_data_c['lunch_id'].isin(test_ids)]

In [43]:
### X_train,y_train, X_test, y_test
features = ['minute', 'last_5_min', 'last_10_min']
X_train = train[features]
y_train = train['next_10_min']
X_test = test[features]
y_test = test['next_10_min']


In [46]:
lunch_info.head(10)

,lunch_id,type
0,slow1,slow
110,slow2,slow
220,slow3,slow
330,slow4,slow
440,slow5,slow
550,slow6,slow
660,slow7,slow
770,slow8,slow
880,slow9,slow
990,slow10,slow


In [44]:
### Overview of X_train 
X_train.tail(5)

,minute,last_5_min,last_10_min
32995,106,34.0,61.0
32996,107,39.0,63.0
32997,108,37.0,63.0
32998,109,30.0,65.0
32999,110,35.0,69.0


### Predicting the students arrival per minute 

In [51]:
### training the model 
model.fit(X_train,y_train)

/opt/anaconda3/envs/miniproject/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [15:26:47] WARNING: /Users/ec2-user/croot/xgboost-split_1779289786370/work/src/learner.cc:782: 
Parameters: { "max_depths" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,1.0
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [ ]:
## Predicting model on each minute 
predicted_output = []
for i in range(lunch_duration):
    every_min_data = data[data['minutes'] == i+1]
    minutes_of_arrival = every_min_data['minutes'].loc[0]
    last_5_mins = every_min_data['last_5_min'].loc[0]
    last_10_mins = every_min_data['last_10_min'].loc[0]


   minutes  arriving_students  last_5_min  last_10_min
0        1                  6         6.0          6.0
   minutes  arriving_students  last_5_min  last_10_min
1        2                  1         7.0          7.0
   minutes  arriving_students  last_5_min  last_10_min
2        3                  2         9.0          9.0
   minutes  arriving_students  last_5_min  last_10_min
3        4                  7        16.0         16.0
   minutes  arriving_students  last_5_min  last_10_min
4        5                  1        17.0         17.0
   minutes  arriving_students  last_5_min  last_10_min
5        6                  7        18.0         24.0
   minutes  arriving_students  last_5_min  last_10_min
6        7                  2        19.0         26.0
   minutes  arriving_students  last_5_min  last_10_min
7        8                  2        19.0         28.0
   minutes  arriving_students  last_5_min  last_10_min
8        9                  7        19.0         35.0
   minutes